In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install flask-cloudflared pyngrok flask

# 1. Install the "Magic Trio" of versions for OpenVLA
#!pip install -q transformers accelerate bitsandbytes sentencepiece
#!pip install transformers accelerate bitsandbytes
#!pip install -q git+https://github.com/huggingface/transformers.git
#!pip install -q accelerate bitsandbytes sentencepiece
!pip install -q accelerate==0.26.0
!pip install -q transformers==4.40.1
!pip install -q timm==0.9.16 bitsandbytes sentencepiece

# 2. Force a restart to flush the old library versions
# import os
# os.kill(os.getpid(), 9)



from flask import Flask, request, jsonify
from pyngrok import ngrok

# 1. Set your Ngrok Token (Get this from ngrok.com)
ngrok.set_auth_token("3DKRH3pBhZww3OudjX7d2UcnLNm_61cuEATtNSFftEzQg5yDf")

app = Flask(__name__)
public_url = ngrok.connect(5000).public_url
print(f" * REMOTE BRAIN IS LIVE AT: {public_url}")

@app.route('/predict', methods=['POST'])
def predict():
    # This is where the OpenVLA model will live
    # For now, we return a 'mock' command to test the bridge
    return jsonify({
        "action": [0.1, 0.0, 0.05, 0, 0, 0, 1] # Move X +0.1, Z +0.05
#    })

app.run()

In [2]:
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq

model_id = "openvla/openvla-7b"

import os

# Point the cache to your Drive
os.environ['HUGGINGFACE_HUB_CACHE'] = '/content/drive/MyDrive/huggingface_cache'

# 1. Load Processor
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)

# 2. Load Model
# Using the downgraded versions, 'load_in_4bit' and 'device_map'
# should now play nicely together.
vla = AutoModelForVision2Seq.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    load_in_4bit=True,
    device_map="auto"
)

print("\n🚀 BRAIN ONLINE! OpenVLA is ready for your Mac.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/130 [00:00<?, ?B/s]

processing_prismatic.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/openvla/openvla-7b:
- processing_prismatic.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


preprocessor_config.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/6.97G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/1.16G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]


🚀 BRAIN ONLINE! OpenVLA is ready for your Mac.


In [ ]:
from flask import Flask, request, jsonify
from pyngrok import ngrok

# 1. Set your Ngrok Token (Get this from ngrok.com)
ngrok.set_auth_token("3DKRH3pBhZww3OudjX7d2UcnLNm_61cuEATtNSFftEzQg5yDf")

import base64
from PIL import Image
from io import BytesIO
import torch

app = Flask(__name__)

@app.route('/')
def home():
    return "Brain is Online", 200

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.json
        img_bytes = base64.b64decode(data['image'])
        img = Image.open(BytesIO(img_bytes)).convert("RGB")


        # prompt = "Pick up the red cube."
        # Dynamically read the prompt from the client request payload
        # Falls back to red cube if the client forgets to include it
        prompt = data.get('prompt', "Pick up the red cube.")
        print(f"🧠 Processing VLA Request with Prompt: '{prompt}'")

        # 1. Process the image and text separately to avoid the 'multiple values' error
        # This is what the processor does under the hood
        #pixel_values = processor.image_processor(img, return_tensors="pt")["pixel_values"]
        #input_ids = processor.tokenizer(prompt, return_tensors="pt")["input_ids"]

        # 2. Combine them into the format the model expects
        #inputs = {
        #    "pixel_values": pixel_values.to("cuda", dtype=torch.bfloat16),
        #    "input_ids": input_ids.to("cuda")
        #}
        """
        print(type(processor))
        print(processor)
        import inspect
        print(inspect.signature(processor.__call__))
        print(processor.image_processor)
        print(processor.tokenizer)
        """

        inputs = processor(images=img, text=prompt, return_tensors="pt")
        print(inputs.keys())

        for k, v in inputs.items():
            print(k, v.shape, v.dtype)

        inputs = {k: v.to("cuda") for k, v in inputs.items()}
        inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)

        # 3. Run Inference
        with torch.inference_mode():
            action = vla.predict_action(**inputs, unnorm_key="bridge_orig", do_sample=False)
            print(action)

        return jsonify({"action": action.tolist()})

    except Exception as e:
        print(f"Error during prediction: {e}")
        return jsonify({"error": str(e)}), 500

# Start Ngrok
print("connecting...")
public_url = ngrok.connect(5000).public_url
print(f" * New Bridge URL: {public_url}/predict")

app.run(port=5000)

connecting...
 * New Bridge URL: https://surface-shuffling-recede.ngrok-free.dev/predict
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [05/Aug/2026 22:01:48] "GET / HTTP/1.1" 200 -


🧠 Processing VLA Request with Prompt: 'Pick up the red cube.'
dict_keys(['input_ids', 'attention_mask', 'pixel_values'])
input_ids torch.Size([1, 7]) torch.int64
attention_mask torch.Size([1, 7]) torch.int64
pixel_values torch.Size([1, 6, 224, 224]) torch.float32


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:944: UserWarning: inner dimension (4304) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(
INFO:werkzeug:127.0.0.1 - - [05/Aug/2026 22:01:53] "POST /predict HTTP/1.1" 200 -


[ 0.00448837 -0.0111082  -0.00128054 -0.00077748 -0.01350803 -0.02585548
  0.        ]
🧠 Processing VLA Request with Prompt: 'Pick up the red cube.'
dict_keys(['input_ids', 'attention_mask', 'pixel_values'])
input_ids torch.Size([1, 7]) torch.int64
attention_mask torch.Size([1, 7]) torch.int64
pixel_values torch.Size([1, 6, 224, 224]) torch.float32


...

[-0.00132716 -0.00430924  0.00599456 -0.00333185  0.00993724 -0.05647964
  0.        ]
🧠 Processing VLA Request with Prompt: 'Pick up the red cube.'
dict_keys(['input_ids', 'attention_mask', 'pixel_values'])
input_ids torch.Size([1, 7]) torch.int64
attention_mask torch.Size([1, 7]) torch.int64
pixel_values torch.Size([1, 6, 224, 224]) torch.float32


INFO:werkzeug:127.0.0.1 - - [05/Aug/2026 23:06:45] "POST /predict HTTP/1.1" 200 -


[-0.00132716 -0.00204292 -0.00231985 -0.00397045 -0.00011073 -0.01618469
  0.        ]
🧠 Processing VLA Request with Prompt: 'Pick up the red cube.'
dict_keys(['input_ids', 'attention_mask', 'pixel_values'])
input_ids torch.Size([1, 7]) torch.int64
attention_mask torch.Size([1, 7]) torch.int64
pixel_values torch.Size([1, 6, 224, 224]) torch.float32


INFO:werkzeug:127.0.0.1 - - [05/Aug/2026 23:06:48] "POST /predict HTTP/1.1" 200 -


[-1.32715853e-03 -2.04292161e-03 -1.28054491e-03 -4.60904041e-03
 -1.28381626e-02 -6.67113405e-05  0.00000000e+00]
